# 03 — Two-Tower Neural Recommender

Builds a two-tower retrieval model on top of the same `train_dataset` / `test_dataset` used in notebooks 01 and 02.

| | MBA (01) | ALS (02) | Two-Tower (03) |
|---|---|---|---|
| **Approach** | Association rules | Matrix factorisation | Neural dual-encoder |
| **User signal** | Cart items | Implicit interactions | Learned ID embedding |
| **Serving** | Lakebase (rules) | Lakebase (pre-computed) | Lakebase (pre-computed) |
| **Scale** | Spark | Spark + Optuna | Ray Train DDP (2× A10 GPU) |

**Flow:** Data Prep (Spark) → Ray Cluster + UC Read → Model Definition → DDP Training → Eval → Pre-compute Recs → Lakebase → MLflow

## 0 — Setup

In [0]:
%pip install "ray[all]==2.54.0" torch mlflow "databricks-sdk>=0.49.0" --quiet
dbutils.library.restartPython()

In [0]:
catalog = 'users'
schema  = 'jon_cheung'

# Input tables (written by 00_data_preparation)
train_table     = f'{catalog}.{schema}.train_dataset'
test_table      = f'{catalog}.{schema}.test_dataset'
item_cat_table  = f'{catalog}.{schema}.item_catalog'
user_feat_table = f'{catalog}.{schema}.user_features'

# Processed tables written by this notebook
interactions_train_table = f'{catalog}.{schema}.two_tower_interactions_train'
interactions_test_table  = f'{catalog}.{schema}.two_tower_interactions_test'
user_vocab_table         = f'{catalog}.{schema}.two_tower_user_vocab'
item_vocab_table         = f'{catalog}.{schema}.two_tower_item_vocab'
recs_table               = f'{catalog}.{schema}.two_tower_recommendations'

# Model registry & experiment
experiment_name = '/Users/jon.cheung@databricks.com/two_tower_recommender'
model_name      = f'{catalog}.{schema}.two_tower_recommender'

# Lakebase — shared project from notebooks 01/02
lakebase_project_id = 'pizza-chain-recommender'
postgres_database   = 'databricks_postgres'

# Feature column names — used in interaction tables, Ray Data, and tower forward pass
USER_FEAT_COLS = [
    'u_total_orders', 'u_avg_order_size', 'u_lifetime_spend', 'u_days_since_last',
    'u_pref_cat_enc', 'u_tier_enc', 'u_recency_enc',
]
ITEM_FEAT_COLS = ['i_base_price', 'i_is_vegetarian', 'i_category_enc', 'i_calories_enc']

NUM_USER_FEATURES = len(USER_FEAT_COLS)   # 7
NUM_ITEM_FEATURES = len(ITEM_FEAT_COLS)   # 4

# Two-tower hyperparameters
embed_dim   = 64
hidden_dim  = 128
num_epochs  = 5
batch_size  = 1024
lr          = 1e-3
temperature = 0.07
k           = 10

## 1 — Data Preparation (Spark)

Explode `order_product_list` → one `(mpid, item)` row per interaction. Fit `StringIndexer` vocab on training data only; apply to test with `handleInvalid='skip'` to handle cold-start gracefully.

> **Parking Lot:** Item features (category, price, popularity) can be joined here once the menu catalog is available in UC. See `PARKING_LOT.md`.

In [0]:
from pyspark.sql.functions import explode, col, when
from pyspark.ml.feature import StringIndexer
import pandas as pd

train = spark.read.table(train_table)
test  = spark.read.table(test_table)
print(f'{train.count():,} training orders  |  {test.count():,} test orders')

# Load feature tables
item_cat_sdf  = spark.read.table(item_cat_table)
user_feat_sdf = spark.read.table(user_feat_table)

def explode_orders(sdf):
    return (
        sdf
        .withColumn('mpid', col('mpid').cast('string'))
        .withColumn('item', explode(col('order_product_list')))
        .select('mpid', 'item')
    )

raw_train = explode_orders(train)
raw_test  = explode_orders(test)

# Fit vocab on train only
user_indexer = StringIndexer(inputCol='mpid', outputCol='user_idx', handleInvalid='skip').fit(raw_train)
item_indexer = StringIndexer(inputCol='item', outputCol='item_idx', handleInvalid='skip').fit(raw_train)

# Encoding maps — must match the feature tensor construction at eval/serve time
category_map = {'pizza': 0, 'side': 1, 'drink': 2, 'dessert': 3}
calories_map = {'low': 0, 'medium': 1, 'high': 2}
tier_map     = {'Bronze': 0, 'Silver': 1, 'Gold': 2}
recency_map  = {'Active': 0, 'Lapsed': 1, 'Churned': 2}

def map_col(column, mapping):
    """Build a when/otherwise chain for label encoding."""
    expr = None
    for label, idx in mapping.items():
        expr = when(col(column) == label, idx) if expr is None else expr.when(col(column) == label, idx)
    return expr.otherwise(0).cast('float')

# Enrich item catalog with encoded features
item_feats_sdf = (
    item_cat_sdf
    .withColumn('i_category_enc',  map_col('category',        category_map))
    .withColumn('i_calories_enc',  map_col('calories_bucket', calories_map))
    .withColumn('i_is_vegetarian', col('is_vegetarian').cast('float'))
    .withColumnRenamed('base_price', 'i_base_price')
    .select('item', 'i_base_price', 'i_is_vegetarian', 'i_category_enc', 'i_calories_enc')
)

# Enrich user features with encoded features
user_feats_sdf = (
    user_feat_sdf
    .withColumn('u_pref_cat_enc', map_col('preferred_category', category_map))
    .withColumn('u_tier_enc',     map_col('customer_tier',      tier_map))
    .withColumn('u_recency_enc',  map_col('recency_bucket',     recency_map))
    .withColumnRenamed('total_orders',          'u_total_orders')
    .withColumnRenamed('avg_order_size',         'u_avg_order_size')
    .withColumnRenamed('lifetime_spend',         'u_lifetime_spend')
    .withColumnRenamed('days_since_last_order',  'u_days_since_last')
    .select('mpid', 'u_total_orders', 'u_avg_order_size', 'u_lifetime_spend',
            'u_days_since_last', 'u_pref_cat_enc', 'u_tier_enc', 'u_recency_enc')
)

def index_and_enrich(sdf):
    return (
        item_indexer.transform(user_indexer.transform(sdf))
        .select('mpid', 'item',
                col('user_idx').cast('long'),
                col('item_idx').cast('long'))
        .join(item_feats_sdf,  on='item', how='left')
        .join(user_feats_sdf,  on='mpid', how='left')
        .select('user_idx', 'item_idx', *USER_FEAT_COLS, *ITEM_FEAT_COLS)
        .fillna(0.0)
    )

interactions_train = index_and_enrich(raw_train)
interactions_test  = index_and_enrich(raw_test)

num_users = len(user_indexer.labels)
num_items = len(item_indexer.labels)

print(f'Vocab  — users: {num_users:,}  |  items: {num_items:,}')
print(f'Pairs  — train: {interactions_train.count():,}  |  test (warm only): {interactions_test.count():,}')
display(interactions_train.limit(5))

In [0]:
# Save interaction tables and vocab mappings to UC
interactions_train.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(interactions_train_table)
interactions_test.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(interactions_test_table)

# Vocab tables: integer index → original label (needed at eval/serve time)
user_vocab_pdf = pd.DataFrame({'user_idx': range(num_users), 'mpid': user_indexer.labels})
item_vocab_pdf = pd.DataFrame({'item_idx': range(num_items), 'item': item_indexer.labels})

spark.createDataFrame(user_vocab_pdf).write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(user_vocab_table)
spark.createDataFrame(item_vocab_pdf).write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(item_vocab_table)

print('Saved to UC:')
for t in [interactions_train_table, interactions_test_table, user_vocab_table, item_vocab_table]:
    print(f'  {t}')

In [ ]:
from pyspark.sql.functions import expr, size, col as spark_col

# Collect all Spark DataFrames to driver memory NOW — Spark workers will be
# unavailable after setup_ray_cluster allocates them to Ray.
# Vocab DataFrames were already built as pandas in cell above.
item_vocab_pd = item_vocab_pdf.copy()
user_vocab_pd = user_vocab_pdf.copy()

# Feature tables: already encoded by map_col() in Section 1 — no re-encoding needed
item_feats_pd = item_feats_sdf.toPandas()
user_feats_pd = (
    user_feats_sdf
    .withColumn('mpid', spark_col('mpid').cast('string'))
    .toPandas()
)

# Test labels for leave-one-out eval: last item per order
test_orders_pd = (
    test
    .filter(size(spark_col('order_product_list')) > 1)
    .withColumn('label', expr('order_product_list[size(order_product_list) - 1]'))
    .withColumn('mpid',  spark_col('mpid').cast('string'))
    .select('mpid', 'label')
    .toPandas()
)

print(f'Collected to driver — items: {len(item_feats_pd)}, users: {len(user_feats_pd)}, test orders: {len(test_orders_pd)}')


## 2 — Ray Cluster Setup + Data Loading

Starts a Ray-on-Spark cluster with 2 GPU workers (1× A10 each).

> **Cluster config required:** Set `spark.task.resource.gpu.amount = 0` in the cluster's Spark config so Ray controls GPU allocation.
> `%pip` installs above must complete before `setup_ray_cluster` — installing after init shuts the cluster down.

In [0]:
import os, ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass
try:
    ray.shutdown()
except Exception:
    pass

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
ray_logs_path = f'/dbfs/Users/{user}/ray_logs/two_tower/'

# Set BEFORE setup_ray_cluster so workers inherit MLflow credentials (required Ray 2.41+)
os.environ['DATABRICKS_HOST']  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ['DATABRICKS_TOKEN'] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# TODO: set num_cpus_worker_node to match your A10 worker node's vCPU count
#   AWS g5.xlarge → 4  |  g5.2xlarge → 8  |  g5.12xlarge → 48
#   Azure Standard_NV36ads_A10_v5 → 36
setup_ray_cluster(
    min_worker_nodes=4,
    max_worker_nodes=4,
    num_cpus_worker_node=8,
    num_gpus_worker_node=0,
    collect_log_to_path=ray_logs_path,
)
print('Ray cluster ready')

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

WAREHOUSE_NAME = 'ray-read-warehouse'
w = WorkspaceClient()
warehouse_id = None

for wh in w.warehouses.list():
    if wh.name == WAREHOUSE_NAME:
        warehouse_id = wh.id
        if wh.state in (State.STOPPED, State.STOPPING):
            w.warehouses.start(wh.id).result()
        print(f'Using warehouse: {WAREHOUSE_NAME} ({warehouse_id})')
        break

if warehouse_id is None:
    created = w.warehouses.create(
        name=WAREHOUSE_NAME, cluster_size='Small',
        auto_stop_mins=10, enable_serverless_compute=True,
        min_num_clusters=1, max_num_clusters=1,
    ).result()
    warehouse_id = created.id
    print(f'Created warehouse: {WAREHOUSE_NAME} ({warehouse_id})')

feat_cols_sql = ', '.join(USER_FEAT_COLS + ITEM_FEAT_COLS)
ray_train_ds  = ray.data.read_databricks_tables(
    warehouse_id=warehouse_id,
    catalog=catalog,
    schema=schema,
    query=f'SELECT user_idx, item_idx, {feat_cols_sql} FROM {interactions_train_table}',
)
print(f'Ray dataset: {ray_train_ds.count():,} interactions')
print(ray_train_ds.schema())

In [ ]:
# Split BEFORE launching Tune — fixed seed guarantees all trials see identical partitions.
# Ray Data streams lazily through each trial; no need to materialize.
ray_train_split, ray_val_split = ray_train_ds.train_test_split(
    test_size=0.2, shuffle=True, seed=42
)
print(f'Train: {ray_train_split.count():,}  |  Val: {ray_val_split.count():,}')


## 3 — Model Definition

Both towers share the same architecture: `Embedding → MLP → L2-normalize`. L2-normalizing outputs makes the dot product equivalent to cosine similarity and keeps magnitudes stable during DDP training.

**In-batch softmax loss:** for a batch of B (user, item) positive pairs, the logit matrix is `[B, B]` — every item in the batch acts as a negative for every other user. Labels are the diagonal. No explicit negative sampling needed.

> **Parking Lot:** Once item features are available, replace `Tower.embedding` with a feature MLP that takes the concatenated feature vector. See `PARKING_LOT.md`.

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UserTower(nn.Module):
    """Embeds a user ID and concatenates side-channel user features before the MLP."""

    def __init__(self, num_users: int, embed_dim: int, num_features: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_users, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim + num_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, user_ids: torch.Tensor, user_feats: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(user_ids)               # [B, embed_dim]
        x   = torch.cat([emb, user_feats], dim=-1)  # [B, embed_dim + num_user_features]
        return F.normalize(self.mlp(x), p=2, dim=-1)


class ItemTower(nn.Module):
    """Embeds an item ID and concatenates side-channel item features before the MLP."""

    def __init__(self, num_items: int, embed_dim: int, num_features: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_items, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim + num_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, item_ids: torch.Tensor, item_feats: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(item_ids)               # [B, embed_dim]
        x   = torch.cat([emb, item_feats], dim=-1)  # [B, embed_dim + num_item_features]
        return F.normalize(self.mlp(x), p=2, dim=-1)


class TwoTowerModel(nn.Module):
    def __init__(self, num_users: int, num_items: int, embed_dim: int,
                 num_user_features: int, num_item_features: int, hidden_dim: int):
        super().__init__()
        self.user_tower = UserTower(num_users, embed_dim, num_user_features, hidden_dim)
        self.item_tower = ItemTower(num_items, embed_dim, num_item_features, hidden_dim)

    def forward(self, user_ids: torch.Tensor, user_feats: torch.Tensor,
                item_ids: torch.Tensor, item_feats: torch.Tensor):
        return self.user_tower(user_ids, user_feats), self.item_tower(item_ids, item_feats)

## 4 — Distributed Training (Ray Train DDP)

In [0]:
import tempfile


def train_fn_per_worker(config: dict):
    """Runs on each Ray Train worker. Reports train_loss every epoch;
    also reports val_loss when a 'val' shard is provided (HPO path)."""
    import tempfile
    import torch
    import torch.nn.functional as F
    import ray.train

    # Set before any torch ops so all intra-op parallelism uses the allocated CPUs
    torch.set_num_threads(config.get('num_threads', 1))

    device          = ray.train.torch.get_device()
    u_feat_cols     = config['user_feat_cols']
    i_feat_cols     = config['item_feat_cols']
    all_feat_dtypes = {c: torch.float32 for c in u_feat_cols + i_feat_cols}

    model = TwoTowerModel(
        config['num_users'], config['num_items'],
        config['embed_dim'],
        config['num_user_features'], config['num_item_features'],
        config['hidden_dim'],
    )
    model     = ray.train.torch.prepare_model(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    train_shard = ray.train.get_dataset_shard('train')
    val_shard   = ray.train.get_dataset_shard('val')   # None for single-run path

    for epoch in range(config['num_epochs']):
        # ── Training ──
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_shard.iter_torch_batches(
            batch_size=config['batch_size'],
            dtypes={'user_idx': torch.long, 'item_idx': torch.long, **all_feat_dtypes},
        ):
            user_ids   = batch['user_idx'].to(device)
            item_ids   = batch['item_idx'].to(device)
            user_feats = torch.stack([batch[c] for c in u_feat_cols], dim=-1).to(device)
            item_feats = torch.stack([batch[c] for c in i_feat_cols], dim=-1).to(device)
            user_emb, item_emb = model(user_ids, user_feats, item_ids, item_feats)
            logits = torch.matmul(user_emb, item_emb.T) / config['temperature']
            labels = torch.arange(len(user_ids), device=device)
            loss   = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        avg_train_loss = total_loss / max(n_batches, 1)

        # ── Validation (HPO path only) ──
        avg_val_loss = None
        if val_shard is not None:
            model.eval()
            val_loss, val_batches = 0.0, 0
            with torch.no_grad():
                for batch in val_shard.iter_torch_batches(
                    batch_size=config['batch_size'],
                    dtypes={'user_idx': torch.long, 'item_idx': torch.long, **all_feat_dtypes},
                ):
                    user_ids   = batch['user_idx'].to(device)
                    item_ids   = batch['item_idx'].to(device)
                    user_feats = torch.stack([batch[c] for c in u_feat_cols], dim=-1).to(device)
                    item_feats = torch.stack([batch[c] for c in i_feat_cols], dim=-1).to(device)
                    user_emb, item_emb = model(user_ids, user_feats, item_ids, item_feats)
                    logits = torch.matmul(user_emb, item_emb.T) / config['temperature']
                    labels = torch.arange(len(user_ids), device=device)
                    val_loss   += F.cross_entropy(logits, labels).item()
                    val_batches += 1
            avg_val_loss = val_loss / max(val_batches, 1)

        metrics = {'epoch': epoch + 1, 'train_loss': avg_train_loss}
        if avg_val_loss is not None:
            metrics['val_loss'] = avg_val_loss

        if ray.train.get_context().get_world_rank() == 0:
            with tempfile.TemporaryDirectory() as tmpdir:
                torch.save(model.module.state_dict(), f'{tmpdir}/model.pt')
                ray.train.report(
                    metrics,
                    checkpoint=ray.train.Checkpoint.from_directory(tmpdir),
                )
        else:
            ray.train.report(metrics)


In [0]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig
from ray.air.integrations.mlflow import MLflowLoggerCallback
import mlflow

train_config = {
    'num_users':         num_users,
    'num_items':         num_items,
    'embed_dim':         embed_dim,
    'num_user_features': NUM_USER_FEATURES,
    'num_item_features': NUM_ITEM_FEATURES,
    'user_feat_cols':    USER_FEAT_COLS,
    'item_feat_cols':    ITEM_FEAT_COLS,
    'hidden_dim':        hidden_dim,
    'num_epochs':        num_epochs,
    'batch_size':        batch_size,
    'lr':                lr,
    'temperature':       temperature,
    'num_threads':       7,
}

mlflow.set_experiment(experiment_name)

trainer = TorchTrainer(
    train_loop_per_worker=train_fn_per_worker,
    train_loop_config=train_config,
    scaling_config=ScalingConfig(num_workers=4, 
                                 resources_per_worker={'CPU':7}, use_gpu=False),
    datasets={'train': ray_train_ds},
    run_config=RunConfig(
        storage_path=f'/dbfs/Users/{user}/two_tower_checkpoints/'
    ),
)

result = trainer.fit()
print(f'Best checkpoint : {result.checkpoint}')
print(f'Final metrics   : {result.metrics}')

## 4b — Hyperparameter Optimisation (Ray Tune + OptunaSearch)

Searches over `lr`, `temperature`, `hidden_dim`, and `embed_dim` using Bayesian optimisation.
Each trial runs a full DDP training job via `TorchTrainer`. `MLflowLoggerCallback` is placed
on the **`Tuner`'s `RunConfig`** — not the `TorchTrainer`'s — as Ray Train v2 only accepts
`ray.train.UserCallback` in the trainer's `RunConfig`.


In [ ]:
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune import TuneConfig, Tuner
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig
from ray.air.integrations.mlflow import MLflowLoggerCallback
import mlflow

# Each trial: num_workers × cpus_per_worker CPUs consumed
# max_concurrent_trials prevents GPU/CPU contention across trials
num_hpo_workers       = 4
cpus_per_worker       = 7
total_cluster_cpus    = 4 * 8   # num_worker_nodes × num_cpus_worker_node
max_concurrent_trials = max(1, total_cluster_cpus // (num_hpo_workers * cpus_per_worker))
print(f'max_concurrent_trials: {max_concurrent_trials}')

# Fixed config shared across all trials
base_config = {
    'num_users':         num_users,
    'num_items':         num_items,
    'num_user_features': NUM_USER_FEATURES,
    'num_item_features': NUM_ITEM_FEATURES,
    'user_feat_cols':    USER_FEAT_COLS,
    'item_feat_cols':    ITEM_FEAT_COLS,
    'num_epochs':        num_epochs,
    'batch_size':        batch_size,
    'num_threads':       cpus_per_worker,
}

# Tunable hyperparameters overlaid on base_config
param_space = {
    **base_config,
    'lr':          tune.loguniform(1e-4, 1e-2),
    'temperature': tune.uniform(0.05, 0.2),
    'hidden_dim':  tune.choice([64, 128, 256]),
    'embed_dim':   tune.choice([32, 64, 128]),
}


def train_driver_fn(config, train_dataset, val_dataset):
    """DDP driver called once per Tune trial. Passes datasets to TorchTrainer
    so Ray Train shards them across workers — do NOT use ray.put() here."""
    trainer = TorchTrainer(
        train_loop_per_worker=train_fn_per_worker,
        train_loop_config=config,
        scaling_config=ScalingConfig(
            num_workers=num_hpo_workers,
            resources_per_worker={'CPU': cpus_per_worker},
            use_gpu=False,
        ),
        datasets={'train': train_dataset, 'val': val_dataset},
        run_config=RunConfig(
            storage_path=f'/dbfs/Users/{user}/two_tower_hpo_checkpoints/',
        ),
    )
    trainer.fit()


mlflow.set_experiment(experiment_name)

tuner = Tuner(
    trainable=tune.with_parameters(
        train_driver_fn,
        train_dataset=ray_train_split,
        val_dataset=ray_val_split,
    ),
    param_space=param_space,
    tune_config=TuneConfig(
        metric='val_loss',
        mode='min',
        search_alg=OptunaSearch(metric='val_loss', mode='min'),
        num_samples=5,
        max_concurrent_trials=max_concurrent_trials,
    ),
    run_config=RunConfig(
        storage_path=f'/dbfs/Users/{user}/two_tower_hpo/',
        callbacks=[
            MLflowLoggerCallback(
                experiment_name=experiment_name,
                save_artifact=True,
                log_params_on_trial_end=True,
            )
        ],
    ),
)

hpo_results = tuner.fit()
best_result = hpo_results.get_best_result(metric='val_loss', mode='min')
print(f'Best val_loss : {best_result.metrics["val_loss"]:.4f}')
print(f'Best config   : {best_result.config}')


## 5 — Evaluation

Leave-one-out protocol matching notebooks 01 and 02: hold out the last item in each test order and measure Hit@K and NDCG@K.

Warm users/items only (cold-start excluded). Scores computed in a single batch matrix multiply — no row-level loops.

In [0]:
import numpy as np
import torch

# Load best checkpoint
with result.checkpoint.as_directory() as ckpt_dir:
    state_dict = torch.load(f'{ckpt_dir}/model.pt', map_location='cpu')

model_eval = TwoTowerModel(num_users, num_items, embed_dim,
                           NUM_USER_FEATURES, NUM_ITEM_FEATURES, hidden_dim)
model_eval.load_state_dict(state_dict)
model_eval.eval()

# ── Build feature tensors from pre-collected pandas (no Spark) ──
# Item: join vocab → already-encoded features, sort by item_idx
_item_merged = (
    item_vocab_pd
    .merge(item_feats_pd, on='item', how='left')
    .sort_values('item_idx')
)
item_feats_tensor = torch.tensor(
    _item_merged[ITEM_FEAT_COLS].fillna(0).values, dtype=torch.float32
)

# User: join vocab → already-encoded features, sort by user_idx
_user_merged = (
    user_vocab_pd
    .assign(mpid=lambda d: d['mpid'].astype(str))
    .merge(user_feats_pd, on='mpid', how='left')
    .sort_values('user_idx')
)
user_feats_tensor = torch.tensor(
    _user_merged[USER_FEAT_COLS].fillna(0).values, dtype=torch.float32
)

# Precompute all item embeddings [num_items, embed_dim]
with torch.no_grad():
    all_item_embs = model_eval.item_tower(torch.arange(num_items), item_feats_tensor)

# ── Leave-one-out evaluation — pure pandas/numpy/torch, no Spark ──
item_name_to_idx = item_vocab_pd.set_index('item')['item_idx'].to_dict()
user_mpid_to_idx = user_vocab_pd.set_index('mpid')['user_idx'].to_dict()

eval_df = test_orders_pd.copy()
eval_df['user_idx']  = eval_df['mpid'].map(user_mpid_to_idx)
eval_df['label_idx'] = eval_df['label'].map(item_name_to_idx)
eval_df = eval_df.dropna(subset=['user_idx', 'label_idx'])
eval_df['user_idx']  = eval_df['user_idx'].astype(int)
eval_df['label_idx'] = eval_df['label_idx'].astype(int)

user_id_tensor   = torch.tensor(eval_df['user_idx'].values)
user_feats_batch = user_feats_tensor[user_id_tensor]

with torch.no_grad():
    user_embs = model_eval.user_tower(user_id_tensor, user_feats_batch)
    scores    = user_embs @ all_item_embs.T

top_k_indices = scores.topk(k, dim=1).indices.numpy()
label_indices = eval_df['label_idx'].values

hits  = [int(lbl in topk) for lbl, topk in zip(label_indices, top_k_indices)]
ranks = [list(topk).index(lbl) + 1 if lbl in topk else None
         for lbl, topk in zip(label_indices, top_k_indices)]
ndcgs = [1 / np.log2(r + 1) if r else 0.0 for r in ranks]

print(f'Evaluation on {len(hits):,} warm test users')
print(f'  Hit@{k}  : {np.mean(hits):.4f}')
print(f'  NDCG@{k} : {np.mean(ndcgs):.4f}')


## 6 — Pre-compute Recommendations + Lakebase

Generates top-K item names per user, saves to UC as `two_tower_recommendations`, then syncs to Lakebase via the same REST API pattern as notebooks 01 and 02.

> **Parking Lot:** Replace with Vector Search ANN once real-time personalisation is needed. See `PARKING_LOT.md`.

In [0]:
import pandas as pd

item_idx_to_name = item_vocab_pd.set_index('item_idx')['item'].to_dict()
user_idx_to_mpid = user_vocab_pd.set_index('user_idx')['mpid'].to_dict()

# Score all users against all items — feature tensors carried over from Section 5
all_user_ids = torch.arange(num_users)
with torch.no_grad():
    all_user_embs  = model_eval.user_tower(all_user_ids, user_feats_tensor)
    all_scores     = all_user_embs @ all_item_embs.T   # all_item_embs from Section 5

top_k_idx_all    = all_scores.topk(k, dim=1).indices.numpy()
top_k_scores_all = all_scores.topk(k, dim=1).values.numpy()

recs_rows = [
    {
        'mpid':            int(float(user_idx_to_mpid[u])),
        'recommendations': [item_idx_to_name[int(i)] for i in top_k_idx_all[u]],
        'scores':          [float(s) for s in top_k_scores_all[u]],
    }
    for u in range(num_users)
]

recs_sdf = spark.createDataFrame(pd.DataFrame(recs_rows))
recs_sdf.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(recs_table)

print(f'Saved {recs_sdf.count():,} user recommendations to {recs_table}')
display(recs_sdf.limit(5))

In [0]:
import time

project_name = f'projects/{lakebase_project_id}'

# Get or create Lakebase project (shared with notebooks 01 and 02)
projects_resp     = w.api_client.do('GET', '/api/2.0/postgres/projects')
existing_projects = {p['name'] for p in projects_resp.get('projects', [])}

if project_name in existing_projects:
    print(f'Lakebase project already exists: {project_name}')
else:
    print(f'Creating Lakebase project: {project_name} ...')
    op = w.api_client.do(
        'POST', '/api/2.0/postgres/projects',
        query={'project_id': lakebase_project_id},
        body={'spec': {'display_name': 'Pizza Chain Recommender', 'pg_version': 17}},
    )
    while not op.get('done'):
        time.sleep(5)
        op = w.api_client.do('GET', f"/api/2.0/postgres/{op.get('name')}")
    print(f'Lakebase project created: {project_name}')

# Enable CDF + PK on mpid (same pattern as notebook 02)
recs_table_short = recs_table.split('.')[-1]
pk_constraint    = f'{recs_table_short}_pk'

spark.sql(f"ALTER TABLE {recs_table} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql(f"ALTER TABLE {recs_table} ALTER COLUMN mpid SET NOT NULL")
spark.sql(f"ALTER TABLE {recs_table} DROP CONSTRAINT IF EXISTS {pk_constraint}")
spark.sql(f"ALTER TABLE {recs_table} ADD CONSTRAINT {pk_constraint} PRIMARY KEY (mpid)")
print(f'PK constraint added: {pk_constraint}')

# Create synced table via REST API
synced_table_id = f'{catalog}.{schema}.two_tower_recommendations_synced'
branch          = f'{project_name}/branches/production'

op = w.api_client.do(
    'POST', '/api/2.0/postgres/synced_tables',
    query={'synced_table_id': synced_table_id},
    body={'spec': {
        'source_table_full_name': recs_table,
        'project':                project_name,
        'branch':                 branch,
        'primary_key_columns':    ['mpid'],
        'scheduling_policy':      'TRIGGERED',
        'postgres_database':      postgres_database,
        'create_database_objects_if_missing': True,
    }},
)
print(f'Synced table operation started: {op.get("name")}')

for _ in range(60):
    if op.get('done'):
        break
    time.sleep(5)
    op = w.api_client.do('GET', f'/api/2.0/postgres/{op.get("name")}')
    print(f'  {op.get("metadata", {}).get("status", "running")}...')

print(f'Synced table ready: {synced_table_id}' if op.get('done') else 'Still syncing — check Lakebase UI')

## 7 — MLflow Logging

Logs the trained PyTorch weights and a Lakebase-backed pyfunc for serving. The pyfunc contract matches the ALS notebook: `mpid` in → `recommendations + scores` out.

In [0]:
import json, os, psycopg2, mlflow
import mlflow.pytorch
import numpy as np

os.makedirs('artifacts', exist_ok=True)

# Look up endpoint (shared project with notebooks 01/02)
branches          = list(w.postgres.list_branches(parent=f'projects/{lakebase_project_id}'))
production_branch = next(b for b in branches if 'production' in b.name)
endpoints         = list(w.postgres.list_endpoints(parent=production_branch.name))
primary_endpoint  = endpoints[0]
print(f'Endpoint: {primary_endpoint.name}')

# Discover actual Postgres schema dynamically (same pattern as notebooks 01/02)
credential = w.postgres.generate_database_credential(endpoint=primary_endpoint.name)
_conn = psycopg2.connect(
    host=primary_endpoint.status.hosts.host, port=5432,
    dbname=postgres_database,
    user=w.current_user.me().user_name,
    password=credential.token,
    sslmode='require',
)
with _conn.cursor() as _cur:
    _cur.execute("""
        SELECT table_schema, table_name FROM information_schema.tables
        WHERE table_name = 'two_tower_recommendations_synced' LIMIT 1
    """)
    _row = _cur.fetchone()
_conn.close()

if not _row:
    raise RuntimeError('two_tower_recommendations_synced not found — has section 6 completed?')
synced_table_pg = f'{_row[0]}.{_row[1]}'
print(f'Synced table in Postgres: {synced_table_pg}')

# Write pyfunc class to a standalone file (avoids cloudpickle issues)
model_file = 'artifacts/two_tower_model.py'
pyfunc_src = 'import json
import pandas as pd
import mlflow.pyfunc
import mlflow


class TwoTowerLakebaseModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import psycopg2
        from databricks.sdk import WorkspaceClient
        with open(context.artifacts["config"]) as f:
            cfg = json.load(f)
        w = WorkspaceClient()
        credential = w.postgres.generate_database_credential(endpoint=cfg["endpoint"])
        self.conn = psycopg2.connect(
            host=cfg["host"], port=5432,
            dbname=cfg["postgres_database"],
            user=cfg["user"],
            password=credential.token,
            sslmode="require",
        )
        self.table = cfg["synced_table"]

    def predict(self, context, model_input):
        results = []
        with self.conn.cursor() as cur:
            for _, row in model_input.iterrows():
                mpid = int(row["mpid"])
                cur.execute(
                    f"""SELECT ARRAY(SELECT jsonb_array_elements_text(recommendations)) AS recommendations,
                               ARRAY(SELECT jsonb_array_elements_text(scores)::float)  AS scores
                        FROM {self.table}
                        WHERE mpid = %s""",
                    (mpid,)
                )
                result = cur.fetchone()
                if result:
                    results.append({"mpid": mpid,
                                    "recommendations": list(result[0]),
                                    "scores": [float(s) for s in result[1]]})
                else:
                    results.append({"mpid": mpid, "recommendations": [], "scores": []})
        return pd.DataFrame(results)


mlflow.models.set_model(TwoTowerLakebaseModel())
'
with open(model_file, 'w') as _f:
    _f.write(pyfunc_src)
print(f'Model file written: {model_file}')

# Config artifact
config = {
    'endpoint':          primary_endpoint.name,
    'host':              primary_endpoint.status.hosts.host,
    'user':              w.current_user.me().user_name,
    'postgres_database': postgres_database,
    'synced_table':      synced_table_pg,
}
with open('artifacts/two_tower_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config written:', config)

# Signature
from mlflow.models.signature import ModelSignature
from mlflow.types import ColSpec, Schema, DataType
from mlflow.types.schema import Array

input_schema  = Schema([ColSpec(DataType.long, 'mpid')])
output_schema = Schema([
    ColSpec(DataType.long,          'mpid'),
    ColSpec(Array(DataType.string), 'recommendations'),
    ColSpec(Array(DataType.double), 'scores'),
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_name='two_tower_lakebase'):
    mlflow.log_params({
        'embed_dim': embed_dim, 'hidden_dim': hidden_dim,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'lr': lr, 'temperature': temperature,
        'num_users': num_users, 'num_items': num_items,
    })
    mlflow.log_metrics({'hit_at_k': float(np.mean(hits)), 'ndcg_at_k': float(np.mean(ndcgs))})
    # PyTorch weights — useful for fine-tuning or embedding extraction
    mlflow.pytorch.log_model(model_eval, name='two_tower_pytorch')
    # Lakebase-backed pyfunc — used for Model Serving
    mlflow.pyfunc.log_model(
        name='model',
        python_model=model_file,
        registered_model_name=model_name,
        artifacts={'config': 'artifacts/two_tower_config.json'},
        pip_requirements=['psycopg2-binary', 'databricks-sdk'],
        signature=signature,
    )
print(f'Model registered as {model_name}')

In [0]:
import mlflow, pandas as pd

# Load the just-registered pyfunc and verify the Lakebase query end-to-end
details   = mlflow.last_active_run()
model_uri = f'runs:/{details.info.run_id}/model'
loaded    = mlflow.pyfunc.load_model(model_uri)

sample_mpids = spark.read.table(recs_table).select('mpid').limit(3).toPandas()
unknown_row  = pd.DataFrame([{'mpid': 999999}])   # should return empty recs
input_data   = pd.concat([sample_mpids, unknown_row], ignore_index=True)

results = loaded.predict(input_data)
for _, row in results.iterrows():
    recs = row['recommendations'][:3]
    print(f"mpid {int(row['mpid'])}: {recs} ...")